# Notebook 4 of 4 — Thresholds, Evaluation, Recommendations

Input: `data/splits.pkl`, `model/model_pipeline.pkl`.

In [ ]:
import warnings
from pathlib import Path
import json
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (classification_report, recall_score, precision_score,
                              f1_score, confusion_matrix)

warnings.filterwarnings("ignore")
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "model"


### Load the split and the trained pipeline

In [ ]:
with open(DATA_DIR / "splits.pkl", "rb") as f:
    X_train, X_test, y_train, y_test, preprocessor = pickle.load(f)

with open(MODEL_DIR / "model_pipeline.pkl", "rb") as f:
    model_pipeline = pickle.load(f)

## 15. The cutoff is a business choice

### 15a. F1-optimal cutoff

The default 0.5 is not a property of this bank. It is a software default. I scan predicted probabilities and keep the cutoff that maximises F1: the best balance of "we caught the leavers" and "the list is not mostly false alarms."

Use this mode when the team wants a stable ranked list and does not have a campaign budget to spend.

In [ ]:
y_prob = model_pipeline.predict_proba(X_test)[:, 1]

best_t, best_f1 = 0.5, -1
for t in np.arange(0.05, 0.95, 0.005):
    f1 = f1_score(y_test, (y_prob >= t).astype(int))
    if f1 > best_f1:
        best_f1, best_t = f1, t

y_pred_f1 = (y_prob >= best_t).astype(int)

print(f"F1-optimal threshold: {best_t:.3f}")
print(f"Recall:    {recall_score(y_test, y_pred_f1):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_f1):.4f}")
print(f"F1:        {f1_score(y_test, y_pred_f1):.4f}")
print()
print(classification_report(y_test, y_pred_f1, target_names=["Stayed", "Churned"]))

### 15b. Cost-optimal cutoff

F1 does not know what a customer is worth. I put two planning assumptions on the cutoff:

- Missing a churner costs about **$5,000** in lifetime value.
- Contacting someone who would have stayed costs about **$10** (an email or a short call).
- The team can contact at most **30%** of the book.

Under those assumptions the cheaper mistake is a false alarm, so the cutoff moves down and recall goes up, until extra contacts stop paying for themselves. These dollar figures are **planning assumptions**, not figures measured in this dataset. If the bank's real save-rate or contact cost is different, the cutoff should be recomputed. The method stays the same.

**Why both modes exist:** day-to-day scoring can sit on the F1 cutoff. A funded retention campaign switches to the cost cutoff and accepts more false alarms, because a $10 contact is a rounding error next to a $5,000 relationship.

In [ ]:
CLV, CAMPAIGN_COST, SAVE_RATE = 5000, 10, 0.30
MAX_CONTACT_RATE = 0.30

n = len(y_test)
thresholds = np.arange(0.05, 0.95, 0.01)
costs, contact_rates = [], []

for t in thresholds:
    yp = (y_prob >= t).astype(int)
    tn = ((yp==0)&(y_test==0)).sum()
    fp = ((yp==1)&(y_test==0)).sum()
    fn = ((yp==0)&(y_test==1)).sum()
    tp = ((yp==1)&(y_test==1)).sum()

    contact_rates.append((tp + fp) / n)
    costs.append(fn*CLV + (tp+fp)*CAMPAIGN_COST - tp*SAVE_RATE*CLV)

costs, contact_rates = np.array(costs), np.array(contact_rates)
feasible = contact_rates <= MAX_CONTACT_RATE
best_idx = np.where(feasible)[0][costs[feasible].argmin()]
best_cost_t = thresholds[best_idx]

y_pred_cost = (y_prob >= best_cost_t).astype(int)

threshold_path = MODEL_DIR / "decision_threshold.json"
threshold_path.write_text(
    json.dumps({"f1_optimal": round(float(best_t), 3), "cost_optimal": round(float(best_cost_t), 3)}),
    encoding="utf-8",
)

print(f"Cost-optimal threshold: {best_cost_t:.3f}")
print(f"Customers contacted:    {contact_rates[best_idx]:.1%} of test set")
print(f"Recall:    {recall_score(y_test, y_pred_cost):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_cost):.4f}")
print(f"F1:        {f1_score(y_test, y_pred_cost):.4f}")
print(f"Est. net cost: ${costs[best_idx]:,.0f}")
print()
print(classification_report(y_test, y_pred_cost, target_names=["Stayed", "Churned"]))

## 16. How to read the confusion matrix

|  | Predicted stay | Predicted leave |
|---|---|---|
| Actually stayed | Correct — no contact needed | False alarm — a cheap contact |
| Actually left | **Missed churner — the expensive error** | Correct — the person the team exists to reach |

A missed churner is the error this project is built to shrink. A false alarm is acceptable when the contact is cheap and the list is capped at what the team can work. The matrix is shown at the **cost** cutoff, because that is the operating point for a campaign.

In [ ]:
cm = confusion_matrix(y_test, y_pred_cost)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Stayed', 'Churned'],
            yticklabels=['Stayed', 'Churned'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

## 17. Why AUC is the ranking check

AUC asks one question: if I pick a churner and a stayer at random, how often does the churner get the higher score? **1.0** is perfect, **0.5** is a coin toss, and **above 0.80** is strong enough to rank a contact list.

AUC does not choose the cutoff. The cost section does that. AUC only tells the business whether the ranking itself is worth using before any threshold is applied.

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

y_prob = model_pipeline.predict_proba(X_test)[:, 1]
auc    = roc_auc_score(y_test, y_prob)

fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f'AUC = {auc:.4f}', color='blue')
plt.plot([0,1], [0,1], 'k--')
plt.title('ROC Curve — Gradient Boosting')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.tight_layout()
plt.show()

## 19. Recommendations for the retention meeting

### Project rationale

I measured who leaves and how much deposit leaves with them, dropped the fields that do not change the decision, encoded the two patterns a straight model would miss (the product cliff, and Germany combined with balance), then refused to pick a model on accuracy. The contact line is set from the cost of a miss versus the cost of a wasted call, under a 30% capacity cap. That sequence is the approach: evidence, then a score, then a budget.

### Where the money is

Of **2,037** exits and about **$186 million** in deposits:

| Priority | Evidence | Action |
|---|---|---|
| Over 41, with a balance | ~45% churn; ~$116m already in the exits from this group | Named relationship coverage and a reviewed rate. Apply the rule to men and women alike |
| Germany | 32.4% churn, ~$120k average balance, ~$98m in exits | A local desk, not the same email used in France and Spain |
| One product, high balance | 1,409 exits, ~$130m | Move them to a **second** product. Do not sell a third |
| Inactive members | 26.9% vs 14.3%; 64% of all exits | Re-activation, then ranked by balance |
| 3–4 products | 83–100% churn, but only ~$25m and 280 exits | **Policy:** cap cross-sell at 2. Case-review the 60 four-product customers; do not build a bank-wide campaign for them |
| Gender gap (25% vs 16%) | Real, and not explained by salary or balance | Service review. Not a gendered offer |
| Credit score, salary, tenure, credit card | Little or no separation | Leave them off the targeting rules |

### How the model supports the retention strategy

The score does not replace those segments. It **ranks inside them**, so a team that can only contact 30% of the book calls the highest expected-loss customers first. F1 mode is the standing list. Cost mode is the mode to switch on when a campaign budget exists.

The lifetime-value and contact-cost numbers are assumptions. Re-run the threshold when finance replaces them with the bank's own figures. The segments above do not depend on those assumptions; they come from the churn rates and the deposits in this book.